#Testing Environment for the Rocket Richard Award, actual code used for the application is found in finalized_scripts/rocketrichard.py

In [5]:
import numpy as np
import pandas as pd
from nhlpy import NHLClient
import ast

In [15]:
#clear csv of empty entries and format into a pandas csv
def clear_csv(csv_path):
    print(csv_path)
    df = pd.read_csv(csv_path, encoding='ascii')
    df = df.dropna()
    return df
rocket_richard = clear_csv("../data/formattedwebscraped/rocketrichard.csv")
rocket_richard

../data/formattedwebscraped/rocketrichard.csv


,szn,winner,runner_up,finalist
1,2025-26,('https://www.nhl.com/player/nathan-mackinnon-...,('https://www.nhl.com/player/cole-caufield-848...,('https://www.nhl.com/player/connor-mcdavid-84...
3,2024-25,('https://www.nhl.com/player/leon-draisaitl-84...,('https://www.nhl.com/player/william-nylander-...,[('https://www.nhl.com/player/alex-ovechkin-84...
5,2023-24,('https://www.nhl.com/player/auston-matthews-8...,('https://www.nhl.com/player/sam-reinhart-8477...,('https://www.nhl.com/player/zach-hyman-847578...
7,2022-23,('https://www.nhl.com/player/connor-mcdavid-84...,('https://www.nhl.com/player/david-pastrnak-84...,('https://www.nhl.com/player/mikko-rantanen-84...
9,2021-22,('https://www.nhl.com/player/auston-matthews-8...,('https://www.nhl.com/player/leon-draisaitl-84...,('https://www.nhl.com/player/chris-kreider-847...
11,2020-21,('https://www.nhl.com/player/auston-matthews-8...,('https://www.nhl.com/player/connor-mcdavid-84...,('https://www.nhl.com/player/alex-debrincat-84...
13,2019-20,[('https://www.nhl.com/player/david-pastrnak-8...,('https://www.nhl.com/player/auston-matthews-8...,('https://www.nhl.com/player/leon-draisaitl-84...
15,2018-19,('https://www.nhl.com/player/alex-ovechkin-847...,('https://www.nhl.com/player/leon-draisaitl-84...,('https://www.nhl.com/player/john-tavares-8475...
17,2017-18,('https://www.nhl.com/player/alex-ovechkin-847...,('https://www.nhl.com/player/patrik-laine-8479...,('https://www.nhl.com/player/william-karlsson-...
19,2016-17,('https://www.nhl.com/player/sidney-crosby-847...,('https://www.nhl.com/player/nikita-kucherov-8...,('https://www.nhl.com/player/auston-matthews-8...


In [88]:
client = NHLClient()
def extractPlayerID(url):

    '''
    purpose:    parses a url and returns that player's id (used by the NHL API) 
    parameters: player url (string)
    returns:    unique player ID (string)
    '''
    #every player ID used by the NHL website and API is 7 digits long
    split = url.rsplit("-")
    return split[-1]

def placeToStats(place_list):
    '''
    purpose:    takes a list of players from the webscraped csv and returns a list of the stats of players in the list
    parameters: place_list (a pandas dataframe of tuples)
    returns:    list of player IDs
    '''
    ids = []
    seasons = place_list.iloc[:,0]
    players = place_list.iloc[:,1]
    seasons = seasons.tolist()
    players = players.tolist()

    for i in range(len(players)):
        playerTuple = ast.literal_eval(players[i])    
        if type(playerTuple) != tuple:      #working with an entry where multiple players won this award
            for entry in playerTuple:
                id = extractPlayerID(entry[0])
                ids.append((seasons[i],id))
            continue
        
        id = extractPlayerID(playerTuple[0])
        ids.append((seasons[i],id))
    return ids

In [ ]:
#extract the player stats of all players in the first, second and third places of the award
first_place = rocket_richard[['szn','winner']]
second_place = rocket_richard[['szn','runner_up']]
third_place = rocket_richard[['szn','finalist']]

first_ids = placeToStats(first_place)
second_ids = placeToStats(second_place)
third_ids = placeToStats(third_place)




[('2025-26', '8477492'),
 ('2024-25', '8477934'),
 ('2023-24', '8479318'),
 ('2022-23', '8478402'),
 ('2021-22', '8479318'),
 ('2020-21', '8479318'),
 ('2019-20', '8477956'),
 ('2019-20', '8471214'),
 ('2018-19', '8471214'),
 ('2017-18', '8471214'),
 ('2016-17', '8471675'),
 ('2015-16', '8471214'),
 ('2014-15', '8471214'),
 ('2013-14', '8471214'),
 ('2012-13', '8471214'),
 ('2011-12', '8474564'),
 ('2010-11', '8470621'),
 ('2008-09', '8471214'),
 ('2007-08', '8471214'),
 ('2006-07', '8467329'),
 ('2005-06', '8467357'),
 ('2002-03', '8460577'),
 ('2000-01', '8455738'),
 ('1999-00', '8455738')]

In [86]:
#Rocket Richard is a statistic directed award, so no models are actually required, just fetch goals of skaters in a given season
#this block of code cuts down the ~700 players in the league to a handful of top scorers

from nhlpy.api.query.builder import QueryBuilder, QueryContext
from nhlpy.api.query.filters.season import SeasonQuery
from nhlpy.api.query.filters.game_type import GameTypeQuery
from nhlpy.api.query.filters.position import PositionQuery, PositionTypes

filters=[
    GameTypeQuery(game_type="2"),                       #2 = regular season, 1=pre-season, 3=playoffs, 4=allstargame
    SeasonQuery("20242025","20242025"),                 #should be put out for an input of the current season
    PositionQuery(position=PositionTypes.ALL_FORWARDS)  
]
query_builder = QueryBuilder()
query_context: QueryContext = query_builder.build(filters=filters)

query_context.fact_query = "gamesPlayed>=1"

data = client.stats.skater_stats_with_query_context(
    report_type='summary',
    query_context=query_context,
    aggregate=True
)

print(len(data['data']))
data['data']

25


[{'assists': 84,
  'evGoals': 29,
  'evPoints': 75,
  'faceoffWinPct': 0.5,
  'gameWinningGoals': 9,
  'gamesPlayed': 78,
  'goals': 37,
  'lastName': 'Kucherov',
  'otGoals': 1,
  'penaltyMinutes': 45,
  'playerId': 8476453,
  'plusMinus': 22,
  'points': 121,
  'pointsPerGame': 1.55128,
  'positionCode': 'R',
  'ppGoals': 8,
  'ppPoints': 46,
  'shGoals': 0,
  'shPoints': 0,
  'shootingPct': 0.13962,
  'shootsCatches': 'L',
  'shots': 265,
  'skaterFullName': 'Nikita Kucherov',
  'timeOnIcePerGame': 1271.3717},
 {'assists': 84,
  'evGoals': 23,
  'evPoints': 78,
  'faceoffWinPct': 0.49933,
  'gameWinningGoals': 5,
  'gamesPlayed': 79,
  'goals': 32,
  'lastName': 'MacKinnon',
  'otGoals': 1,
  'penaltyMinutes': 41,
  'playerId': 8477492,
  'plusMinus': 25,
  'points': 116,
  'pointsPerGame': 1.46835,
  'positionCode': 'C',
  'ppGoals': 9,
  'ppPoints': 38,
  'shGoals': 0,
  'shPoints': 0,
  'shootingPct': 0.1,
  'shootsCatches': 'R',
  'shots': 320,
  'skaterFullName': 'Nathan MacKin

In [87]:
above25goals = data['data']         #based on the filtering above, this is a hard coded 2024-2025 season handful of players that are most likely to win rocket richard
df = pd.DataFrame(above25goals)

#add an extra column of average time on ice across the whole season
df['averageTOI'] = np.zeros(df.shape[0])
df['averageTOI'] = (df['timeOnIcePerGame']/60)
df


,assists,evGoals,evPoints,faceoffWinPct,gameWinningGoals,gamesPlayed,goals,lastName,otGoals,penaltyMinutes,...,ppGoals,ppPoints,shGoals,shPoints,shootingPct,shootsCatches,shots,skaterFullName,timeOnIcePerGame,averageTOI
0,84,29,75,0.50000,9,78,37,Kucherov,1,45,...,8,46,0,0,0.13962,L,265,Nikita Kucherov,1271.3717,21.189528
1,84,23,78,0.49933,5,79,32,MacKinnon,1,41,...,9,38,0,0,0.10000,R,320,Nathan MacKinnon,1366.9493,22.782488
2,54,36,73,0.54347,11,71,52,Draisaitl,6,34,...,16,33,0,0,0.21666,L,240,Leon Draisaitl,1291.0422,21.517370
3,63,34,83,0.37500,4,82,43,Pastrnak,3,42,...,9,23,0,0,0.13479,R,319,David Pastrnak,1211.1707,20.186178
4,75,20,67,0.29411,7,81,27,Marner,3,14,...,6,33,1,2,0.15606,R,173,Mitch Marner,1278.9259,21.315432
5,74,17,69,0.47680,3,67,26,McDavid,0,37,...,9,31,0,0,0.13265,L,196,Connor McDavid,1321.8805,22.031342
6,56,30,67,0.12500,8,82,41,Connor,2,25,...,9,28,2,2,0.15355,L,267,Kyle Connor,1223.6829,20.394715
7,66,22,58,0.45614,3,77,28,Eichel,0,8,...,5,34,1,2,0.12017,R,233,Jack Eichel,1232.0129,20.533548
8,58,21,64,0.57014,10,80,33,Crosby,4,31,...,12,27,0,0,0.14537,L,227,Sidney Crosby,1222.4750,20.374583
9,60,20,53,0.33333,5,81,30,Keller,1,28,...,10,37,0,0,0.13761,L,218,Clayton Keller,1147.0864,19.118107


In [69]:
#recreate the above but for any year
from nhlpy.api.query.builder import QueryBuilder, QueryContext
from nhlpy.api.query.filters.season import SeasonQuery
from nhlpy.api.query.filters.game_type import GameTypeQuery
from nhlpy.api.query.filters.position import PositionQuery, PositionTypes

def predictHistoricalRRWinner(year):
    '''
    purpose:    takes a given season and produces the historical winner of the Rocket Richard award for that given year 
    parameters: year (string) either in yyyy or yyyyyyyy format
    returns:    a tuple (winner name, # of goals, player ID)
    '''
    #format year for the filter (is a string when inputted)
    if len(year) == 4:
        int_year = int(year)
        interval = (int_year,int_year+1)
        year_interval = str(interval[0]) + str(interval[1])
    elif len(year) == 8:    #20252026
        year_interval = year
    else:
        raise SyntaxError("requires yyyy or yyyyyyyy format of season year")
    
    
    filters=[
    GameTypeQuery(game_type="2"),                       #2 = regular season, 1=pre-season, 3=playoffs, 4=allstargame
    SeasonQuery(year_interval,year_interval),                             #should be put out for an input of the current season
    PositionQuery(position=PositionTypes.ALL_FORWARDS)  
    ]
    query_builder = QueryBuilder()
    query_context: QueryContext = query_builder.build(filters=filters)

    query_context.fact_query = "gamesPlayed>=1 and goals >=25"  #this is a hard coded assumption, the lower the goal threshold the more players to work with

    data = client.stats.skater_stats_with_query_context(
        report_type='summary',
        query_context=query_context,
        aggregate=True
    )

    dfa25 = pd.DataFrame(data['data'])      #a dataframe of all players above 25 goals in a given season
    
    #now, "predict" the historical winner of this year
    #since RR is only determined by the highest # of goals, sort as such

    highlight = dfa25[['goals','skaterFullName', 'playerId']]
    standings = highlight.sort_values(by='goals',ascending=False)
    winner = standings.iloc[0]
    return tuple(winner)

    


In [70]:
#tests

print(predictHistoricalRRWinner("2024"))       #enter starting year OR interval year
#print(predictHistoricalRRWinner("20252026"))   #

(np.int64(52), 'Leon Draisaitl', np.int64(8477934))


### Milestone #1: Identify Past Rocket Winners and start a particular dataset with these labels put together

In [66]:
#strip seasons
seasons = rocket_richard['szn']
for szn in seasons:
    nohyphen = szn.split("-")
    szn = nohyphen[0]
    print(szn, predictHistoricalRRWinner(szn))


2025 (np.int64(53), 'Nathan MacKinnon', np.int64(8477492))
2024 (np.int64(52), 'Leon Draisaitl', np.int64(8477934))
2023 (np.int64(69), 'Auston Matthews', np.int64(8479318))
2022 (np.int64(64), 'Connor McDavid', np.int64(8478402))
2021 (np.int64(60), 'Auston Matthews', np.int64(8479318))
2020 (np.int64(41), 'Auston Matthews', np.int64(8479318))
2019 (np.int64(48), 'Alex Ovechkin', np.int64(8471214))
2018 (np.int64(51), 'Alex Ovechkin', np.int64(8471214))
2017 (np.int64(49), 'Alex Ovechkin', np.int64(8471214))
2016 (np.int64(44), 'Sidney Crosby', np.int64(8471675))
2015 (np.int64(50), 'Alex Ovechkin', np.int64(8471214))
2014 (np.int64(53), 'Alex Ovechkin', np.int64(8471214))
2013 (np.int64(51), 'Alex Ovechkin', np.int64(8471214))
2012 (np.int64(32), 'Alex Ovechkin', np.int64(8471214))
2011 (np.int64(60), 'Steven Stamkos', np.int64(8474564))
2010 (np.int64(50), 'Corey Perry', np.int64(8470621))
2008 (np.int64(56), 'Alex Ovechkin', np.int64(8471214))
2007 (np.int64(65), 'Alex Ovechkin', n